# SAR Polarimetric Matrix Speckle Filter Tutorial — Refined Lee

Scalar speckle filters (Lee, ComplexLee) treat each polarization channel independently, which
can distort the inter-channel covariance structure that encodes polarimetric information.
The **Refined Lee filter** operates on the full polarimetric covariance/coherency matrix jointly,
preserving the matrix structure while suppressing speckle.

**Algorithms demonstrated:**
1. Per-channel `ComplexLeeFilter` — independent channel filtering (baseline)
2. `RefinedLeeFilter` — edge-preserving joint matrix filtering

**Data:** NISAR L1 RSLC quad-pol or synthetic fallback.

## Theory — Refined Lee Filter

The Refined Lee filter (Lee et al. 1999) applies the MMSE Lee criterion to every element
of the polarimetric coherency matrix $[T_3]$ simultaneously, using a shared MMSE weight
$b$ derived from the span image:

$$\hat{T}_{ij} = \bar{T}_{ij}^{(d)} + b \left(T_{ij} - \bar{T}_{ij}^{(d)}\right)$$

where $\bar{T}_{ij}^{(d)}$ is the mean within the selected directional mask.

**Directional mask selection:**

1. Compute the *span* $= T_{11} + T_{22} + T_{33}$ (total power).
2. Evaluate 4 directional gradients on $3\times 3$ sub-windows of the span:
   horizontal, 45°, vertical, 135°.
3. Select the direction of maximum gradient magnitude.
4. Use the half-plane mask on the more-homogeneous side for local statistics.
5. Compute $b = \max\bigl(0,\,(c_v^2 - \sigma^2) / (c_v^2(1+\sigma^2))\bigr)$.

This preserves edges across all matrix elements simultaneously, maintaining the
Hermitian positive semi-definite structure required for valid decompositions.

**References:**
- Lee, J.-S., Grunes, M.R., and de Grandi, G. (1999). *Polarimetric SAR speckle filtering
  and its implication for classification.* IEEE Trans. Geoscience and Remote Sensing,
  37(5), pp. 2363–2373.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
import gc

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

try:
    import grdl
except ImportError as exc:
    raise ImportError(
        "\n"
        "═══════════════════════════════════════════════════════════════\n"
        "  grdl is NOT installed in this Python environment.\n"
        "  Install with:  pip install -e '.[all]'\n"
        "  from the grdl repository root.\n"
        "═══════════════════════════════════════════════════════════════\n"
    ) from exc

print(f"grdl {grdl.__version__} — ready")


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from grdl.IO.sar.nisar import NISARReader
from grdl.image_processing.filters import RefinedLeeFilter, ComplexLeeFilter
from grdl.image_processing.decomposition import CoherencyMatrix

try:
    from grdk.viewers.dual_viewer import DualGeoViewer
    HAS_GRDK = True
    try:
        get_ipython().run_line_magic('gui', 'qt6')
    except (NameError, AttributeError):
        pass
except ImportError as exc:
    DualGeoViewer = None
    HAS_GRDK = False
    print(f"Warning: grdk import failed ({exc}). Falling back to matplotlib.")

def show_dual_rgb(left, right, title, left_label='Left', right_label='Right'):
    if HAS_GRDK:
        viewer = DualGeoViewer()
        viewer.set_mode('dual')
        viewer.setWindowTitle(title)
        viewer.set_array(left, pane=0)
        viewer.set_array(right, pane=1)
        viewer.show()
        print(f"GRDK viewer launched: {title}")
        return viewer

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), dpi=100)
    for ax, arr, label in zip(axes, (left, right), (left_label, right_label)):
        img = np.asarray(arr)
        if img.ndim == 3 and img.shape[0] in (1, 3, 4):
            img = np.moveaxis(img, 0, -1)
        if img.ndim == 3 and img.shape[-1] == 1:
            img = img[..., 0]
        ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
        ax.set_title(label)
        ax.axis('off')
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()
    print(f"Matplotlib fallback shown: {title}")
    return fig

print('Imports OK')


In [ ]:
# =============================================================================
# Configuration
# =============================================================================
nisar_file = Path(
    '/data/sar/slc/nisar/l1_rslc/'
    '20260105T055924_20260105T055931/'
    'NISAR_L1_PR_RSLC_009_116_D_054_2005_QPDH_A_20260105T055924_20260105T055931'
    '_X05010_N_P_J_001.h5'
)
frequency    = 'A'
polarizations = 'all'
chip_size    = 1024   # pixels (square)

# Filter parameters
rl_kernel    = 5     # Refined Lee kernel size (odd, 3–31)
rl_nlook     = 1.0   # ENL for Refined Lee
cl_kernel    = 7     # ComplexLee kernel for per-channel baseline
boxcar_win   = 7     # Window for boxcar-averaged T3

In [ ]:
# =============================================================================
# Load NISAR quad-pol chip or generate synthetic data
# =============================================================================
def _synthetic_quad_pol(rows=512, cols=512, seed=42):
    rng = np.random.default_rng(seed)
    shh = (rng.standard_normal((rows, cols)) + 1j * rng.standard_normal((rows, cols))).astype(np.complex64)
    shv = (0.3 * (rng.standard_normal((rows, cols)) + 1j * rng.standard_normal((rows, cols)))).astype(np.complex64)
    svh = shv.copy()
    svv = (rng.standard_normal((rows, cols)) + 1j * rng.standard_normal((rows, cols))).astype(np.complex64)
    return shh, shv, svh, svv

if nisar_file.exists():
    reader = NISARReader(nisar_file, frequency=frequency, polarizations=polarizations)
    meta = reader.metadata
    N = chip_size
    r0 = max(0, (meta.rows - N) // 2)
    c0 = max(0, (meta.cols - N) // 2)
    cube = reader.read_chip(r0, r0 + N, c0, c0 + N)
    reader.close()
    pol_index = {cm.polarization: i for i, cm in enumerate(meta.channel_metadata)}
    shh = cube[pol_index['HH']]
    shv = cube[pol_index['HV']]
    svh = cube[pol_index.get('VH', pol_index['HV'])]
    svv = cube[pol_index['VV']]
    print(f'NISAR chip: {shh.shape}')
    del cube
    gc.collect()

else:
    N = chip_size
    shh, shv, svh, svv = _synthetic_quad_pol(rows=N, cols=N)
    print(f'Synthetic quad-pol: {shh.shape} (no NISAR file at {nisar_file})')

In [ ]:
# =============================================================================
# Display helper
# =============================================================================
def span_to_rgb(t3, pct=(2, 98)):
    """Span (trace of T3) as grayscale (3, rows, cols) display image."""
    span = np.real(t3[0, 0] + t3[1, 1] + t3[2, 2])
    db = 10.0 * np.log10(np.maximum(span, 1e-10))
    lo, hi = np.percentile(db, pct)
    gray = np.clip((db - lo) / max(hi - lo, 1e-8), 0.0, 1.0).astype(np.float32)
    return np.stack([gray, gray, gray], axis=0)

def pol_rgb(shh, shv, svv, pct=(5, 95)):
    """Pauli-colour RGB: R=|HH-VV|, G=|HV|, B=|HH+VV|."""
    r = np.abs(shh - svv)
    g = np.abs(shv)
    b = np.abs(shh + svv)
    def stretch(x):
        lo, hi = np.percentile(x, pct)
        return np.clip((x - lo) / max(hi - lo, 1e-8), 0.0, 1.0).astype(np.float32)
    return np.stack([stretch(r), stretch(g), stretch(b)], axis=0)

print('Display helpers defined')

---
## 1. Build the Boxcar-Averaged Coherency Matrix [T3]

The `CoherencyMatrix` class builds $[T_3]$ from Pauli-basis SLC channels by
spatial averaging within a boxcar window.  This is the standard (non-edge-preserving)
baseline.

In [ ]:
# =============================================================================
# Boxcar-averaged T3
# =============================================================================
channels = np.stack([shh, shv, svh, svv], axis=0)
coh = CoherencyMatrix(window_size=boxcar_win)
t3_boxcar = coh.compute(channels)   # (3, 3, rows, cols)

span_boxcar = np.real(t3_boxcar[0, 0] + t3_boxcar[1, 1] + t3_boxcar[2, 2])
print(f'T3 shape:   {t3_boxcar.shape}')
print(f'Span range: [{span_boxcar.min():.4f}, {span_boxcar.max():.4f}]')

---
## 2. Refined Lee Filter on [T3]

`RefinedLeeFilter.filter_matrix(t3)` accepts the pre-averaged $(3,3,\text{rows},\text{cols})$
matrix and returns a filtered matrix of the same shape.  Alternatively,
`filter_channels(shh, shv, svh, svv)` builds [T3] and filters in one step.

In [ ]:
# =============================================================================
# RefinedLeeFilter — from SLC channels directly
# =============================================================================
import time
rlf = RefinedLeeFilter(kernel_size=rl_kernel, nlook=rl_nlook)

t0 = time.perf_counter()
t3_refined = rlf.filter_channels(shh, shv, svh, svv, matrix_type='T3')
elapsed = time.perf_counter() - t0

span_refined = np.real(t3_refined[0, 0] + t3_refined[1, 1] + t3_refined[2, 2])
print(f'Refined Lee (k={rl_kernel}) applied in {elapsed:.1f}s')
print(f'Filtered span range: [{span_refined.min():.4f}, {span_refined.max():.4f}]')

In [ ]:
# =============================================================================
# GRDK dual-pane viewer — span: boxcar vs Refined Lee
# =============================================================================
viewer_rl = show_dual_rgb(
    span_to_rgb(t3_boxcar),
    span_to_rgb(t3_refined),
    f'Span — Boxcar {boxcar_win}x{boxcar_win} (L) vs RefinedLee k={rl_kernel} (R)',
    left_label='Boxcar span',
    right_label='Refined Lee span',
)
print('Viewer: span comparison')

---
## 3. Per-Channel ComplexLee vs Refined Lee — Pauli RGB

Applying ComplexLeeFilter to each channel independently may preserve total power but
can distort the off-diagonal covariance terms.  Compare the Pauli-colour display.

In [ ]:
# =============================================================================
# Per-channel ComplexLeeFilter baseline
# =============================================================================
clf = ComplexLeeFilter(kernel_size=cl_kernel)
hh_lee = clf.apply(shh)
hv_lee = clf.apply(shv)
vv_lee = clf.apply(svv)

rgb_raw     = pol_rgb(shh,    shv,    svv)
rgb_cl_lee  = pol_rgb(hh_lee, hv_lee, vv_lee)

# For Refined Lee, reconstruct Pauli from filtered T3 diagonal elements
# T3[0,0] = |k0|^2, T3[1,1] = |k1|^2, T3[2,2] = |k2|^2 (Pauli powers)
r_rl = np.sqrt(np.maximum(np.real(t3_refined[1, 1]), 0))  # |double-bounce|
g_rl = np.sqrt(np.maximum(np.real(t3_refined[0, 0]), 0))  # |surface+dbl|
b_rl = np.sqrt(np.maximum(np.real(t3_refined[2, 2]), 0))  # |volume|

def stretch(x, pct=(2, 98)):
    lo, hi = np.percentile(x, pct)
    return np.clip((x - lo) / max(hi - lo, 1e-8), 0.0, 1.0).astype(np.float32)

rgb_rl = np.stack([stretch(r_rl), stretch(g_rl), stretch(b_rl)], axis=0)
print('RGB images ready')

In [ ]:
# =============================================================================
# GRDK dual-pane viewer — per-channel Lee vs Refined Lee (Pauli RGB)
# =============================================================================
viewer_cmp = show_dual_rgb(
    rgb_cl_lee,
    rgb_rl,
    f'Pauli RGB — ComplexLee k={cl_kernel} per-channel (L) vs RefinedLee k={rl_kernel} (R)',
    left_label='ComplexLee per-channel',
    right_label='Refined Lee Pauli RGB',
)
print('Viewer: ComplexLee vs RefinedLee')

---
## Memory Cleanup

Free all large arrays and data structures from memory.

In [ ]:
# =============================================================================
# Memory cleanup — delete all large data structures
# =============================================================================
print("Cleaning up memory...")

# Delete raw quad-pol data
del shh, shv, svh, svv, channels

# Delete cube if it exists (from NISAR reader)
try:
    del cube
except NameError:
    pass

# Delete coherency matrices
del t3_boxcar, t3_refined

# Delete span arrays
del span_boxcar, span_refined

# Delete filtered channels
del hh_lee, hv_lee, vv_lee

# Delete RGB arrays
del rgb_raw, rgb_cl_lee, rgb_rl, r_rl, g_rl, b_rl

# Delete filter instances
del coh, rlf, clf

# Delete viewers
try:
    del viewer_rl, viewer_cmp
except NameError:
    pass

# Force garbage collection
gc.collect()

print("✓ Memory cleanup complete")